# Full 803 x 3 evaluation of chess-coach **v6-dpo2** (resumable, Colab)

This notebook runs the **full 803-position gap benchmark x 3 tiers = 2409 scenarios**
evaluation of the `chess-coach-32b-v6-dpo2` adapter and produces **tier-policy match**
numbers that are directly comparable to `RESULTS_STAGE4_CORRECTED.md` /
`RESULTS_FULL_EVAL_803.md`, so v6-dpo2 can be added to the 803 field.

**Why it is faithful (not a re-implementation):**
- The GROUNDED prompts are **precomputed offline and byte-identical** to the Stage-4 eval
  (verified fact sheet + `render_user_prompt` + the shared format instruction, built from
  the committed corrected-v6 grounding). They live in the HF dataset
  `khoilamalphaai/chess-coach-803-eval-prompts`, so **no chess engine is installed here**.
- Generation uses the **exact Stage-4 decode**: greedy `do_sample=False`,
  `repetition_penalty=1.15`, `no_repeat_ngram_size=4`, `max_new_tokens=256`, over the exact
  base `unsloth/Qwen3-32B-unsloth-bnb-4bit` + the published v6-dpo2 LoRA.
- Scoring uses the **vendored extractor copied verbatim** from the repo
  (`src.eval.evaluate.extract_recommended_move`), against the committed corrected labels.

**Built-in proof:** the 120 held-out TEST positions are a strict subset of the 2409, so this
run reproduces the published Stage-4 v6-dpo2 numbers on that subset (tier-policy **0.8917**,
B/I/A 0.8583/0.8417/0.9750). If the subset does not reproduce, the run is not comparable.

**Hardware:** the faithful 4-bit base is ~38 GB resident, so this needs an **A100-80GB**
runtime. A100-40GB is very tight (small batch, likely OOM); L4/T4 (<=24 GB) cannot hold it.

**Resumable:** every batch is checkpointed to Google Drive. If Colab disconnects or you run
out of compute units, just re-run the cells top to bottom; generation resumes and loses
nothing already written.

**Honesty:** "tier-policy match" is agreement with a preregistered project rule
(learnability), not validated pedagogy. These are corrected-v6 fresh-grounding numbers;
compare them to the corrected-v6 tables, not to the pre-correction v4-era field.

## Step 0 - Check the GPU runtime

Runtime menu -> Change runtime type -> **A100 GPU** (80 GB). Then run this cell.

In [ ]:
import subprocess
out = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip()
print(out or 'nvidia-smi not available - is a GPU runtime selected?')
GPU_MEM_GB = 0.0
try:
    GPU_MEM_GB = int(out.split(',')[1].strip().split()[0]) / 1024.0
except Exception:
    pass
print('Detected ~%.0f GB GPU memory' % GPU_MEM_GB)
if GPU_MEM_GB >= 70:
    RECOMMENDED_BATCH = 24
    print('OK: A100-80GB class. The faithful ~38 GB base fits with KV headroom (batch 24).')
elif GPU_MEM_GB >= 38:
    RECOMMENDED_BATCH = 6
    print('WARNING: ~40 GB GPU. The faithful ~38 GB base barely fits; use a SMALL batch (6) '
          'and expect possible OOM. A100-80GB is strongly recommended.')
else:
    RECOMMENDED_BATCH = 2
    print('WARNING: <38 GB GPU (L4/T4). The faithful 4-bit base is ~38 GB and will NOT fit. '
          'Switch the runtime to A100-80GB (see README_colab.md).')

## Step 1 - Install dependencies (pinned, known-good for Qwen3 on Colab)

Takes a few minutes. Uses the community-validated Unsloth + Qwen3 Colab pin set (transformers 4.57.1).

In [ ]:
%%capture
import os, re, torch
# Community-validated Unsloth + Qwen3 install for Colab (avoids the common bitsandbytes/
# xformers version clashes). Pins transformers==4.57.1 (Qwen3-capable) + trl==0.22.2.
v = re.match(r'[0-9]+\.[0-9]+', str(torch.__version__)).group(0)
xformers = 'xformers==' + {'2.10': '0.0.34', '2.9': '0.0.33.post1', '2.8': '0.0.32.post2'}.get(v, '0.0.34')
!pip install -q sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer python-chess
!pip install -q --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install -q transformers==4.57.1
!pip install -q --no-deps trl==0.22.2

## Step 2 - Configuration

Paste your Hugging Face token (read access is enough - the base, adapter, and prompts are public). Nothing else needs changing for a faithful full run.

In [ ]:
import getpass, os
HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Hugging Face token: ')
os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

# --- Model + data ids (public) ---
BASE_MODEL      = 'unsloth/Qwen3-32B-unsloth-bnb-4bit'   # the EXACT base v6-dpo2 was trained on
ADAPTER_REPO    = 'khoilamalphaai/chess-coach-32b-v6-dpo2'
PROMPTS_DATASET = 'khoilamalphaai/chess-coach-803-eval-prompts'
PROMPTS_FILE    = 'eval803_grounded_prompts.jsonl'

# --- Decode: byte-identical to scripts/stage4_eval_v6dpo2.py. DO NOT change for a comparable run. ---
MAX_SEQ_LEN       = 3072
MAX_NEW_TOKENS    = 256
REPETITION_PENALTY = 1.15
NO_REPEAT_NGRAM    = 4

# --- Run controls ---
RESUME     = True   # skip scenarios already checkpointed to Drive
LIMIT      = 0      # 0 = full 2409; set e.g. 24 for a quick smoke test
EVAL_BATCH = globals().get('RECOMMENDED_BATCH', 24)  # from the GPU-check cell
print('config OK | base=%s | adapter=%s | batch=%d | limit=%s' % (BASE_MODEL, ADAPTER_REPO, EVAL_BATCH, LIMIT or 'full'))

## Step 3 - Mount Google Drive (for resumable checkpoints)

All generations are appended to a file on your Drive after every batch, so a disconnect or credit-out never loses completed work.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
CKPT_DIR   = '/content/drive/MyDrive/chess_coach_803_eval'
os.makedirs(CKPT_DIR, exist_ok=True)
GENS_PATH   = os.path.join(CKPT_DIR, 'gens_v6dpo2_803.jsonl')   # resumable per-scenario generations
SCORES_PATH = os.path.join(CKPT_DIR, 'scores_v6dpo2_803.json')  # final metrics
print('checkpoint dir :', CKPT_DIR)
print('gens (resume)  :', GENS_PATH)

## Step 4 - Download the precomputed prompts + the v6-dpo2 adapter

The ~39 GB 4-bit base is fetched automatically when the model loads (Step 6).

In [ ]:
from huggingface_hub import hf_hub_download, snapshot_download
import json
prompts_path = hf_hub_download(PROMPTS_DATASET, PROMPTS_FILE, repo_type='dataset', token=HF_TOKEN)
ROWS = [json.loads(x) for x in open(prompts_path, encoding='utf-8') if x.strip()]
if LIMIT:
    ROWS = ROWS[:LIMIT]
n_pos = len({r['pos_id'] for r in ROWS})
n_val = sum(1 for r in ROWS if r['is_val'])
print('loaded %d grounded prompts | %d positions | %d held-out TEST' % (len(ROWS), n_pos, n_val))
ADAPTER_DIR = snapshot_download(ADAPTER_REPO, token=HF_TOKEN)
print('adapter at:', ADAPTER_DIR)

## Step 5 - Vendored move extractor + scorer (byte-identical to the repo)

The first cell is copied verbatim from the repo so scores match `scripts/reproduce_v4.py`. The second defines the deterministic metrics exactly as `scripts/stage4_eval_v6dpo2.py`.

In [ ]:
# ============================================================================
# VENDORED MOVE EXTRACTOR - byte-identical to the chess-instructor-llm repo.
# Segments below are copied VERBATIM (extracted via ast.get_source_segment) from:
#   * src/teacher/coach_gate.py : pick_recommendation (+ its regexes / helpers)
#   * src/eval/evaluate.py      : extract_recommended_move (+ _extract_json_object, _legal_uci)
# This is the SAME strict, any-legal extractor scripts/reproduce_v4.py asserts and
# Stage-4 uses, so tier-policy scores here match RESULTS_STAGE4_CORRECTED.md exactly.
# ============================================================================
import json
import re
from typing import Any, Callable, Dict, List, Optional, Sequence, Tuple
import chess

# --- from src/teacher/coach_gate.py (verbatim) ---
_SAN_RE = re.compile(r"(O-O-O|O-O|[KQRBN]?[a-h]?[1-8]?x?[a-h][1-8](?:=[QRBN])?[+#]?)")

_ENDORSE_CUE_RE = re.compile(
    r"(?:i['\u2019]?d\s+play|i\s+would\s+play|i['\u2019]?ll\s+play|i\s+play|"
    r"i\s+recommend|recommend(?:ed)?(?:\s+move)?(?:\s+is)?|"
    r"(?:the\s+)?move\s*(?:is|:)|/move\s*:|best\s+move\s+is|go\s+with|choose|"
    r"leads?\s+you\s+to|points?\s+you\s+to|improvement\s+is|better\s+is)"
    r"\s*[:\-]?\s*",
    re.IGNORECASE,
)

_AVOID_CUE_RE = re.compile(
    r"rather than|instead of|such as|\bavoid\w*|\blike\b|\bnot\b|\bnever\b|"
    r"n['\u2019]?t\b|don['\u2019]?t|do not|rush\w*\s+into|forcing[- ]?looking",
    re.IGNORECASE,
)

_HYPO_CUE_RE = re.compile(
    r"\bif\b|\bafter\b|\bthen\b|for\s+example|follow[- ]?up|followed\s+by|"
    r"you\s+have|you['\u2019]?ll\s+have|\breplies\b|\brespond|\bnext\b|continu|"
    r"(?:white|black)\s+(?:plays|has|goes|replies)",
    re.IGNORECASE,
)

_COORD_BEFORE_RE = re.compile(r"\b(?:on|to)\b\W*$", re.IGNORECASE)

_COORD_AFTER_RE = re.compile(r"^[- ]?(?:pawn|square|file|rank)s?\b", re.IGNORECASE)

_CONCESSION_AFTER_RE = re.compile(
    r"^\W{0,3}(?:was|is|were|are)\s+(?:already\s+|also\s+)?"
    r"(?:playable|possible|fine|active|ok|okay|tempting|reasonable)",
    re.IGNORECASE,
)

_CLAUSE_BOUNDARY_RE = re.compile(r"[.!?:;](?=\s)|\n")

def _clause_before(text: str, start: int, span: int = 90) -> str:
    """The clause immediately preceding ``start`` — everything after the last real
    sentence boundary within ``span`` chars. Used to read a move's framing without
    being fooled by the dots of a chess ellipsis ("...b2+")."""
    pre = text[max(0, start - span) : start]
    last = -1
    for m in _CLAUSE_BOUNDARY_RE.finditer(pre):
        last = m.end()
    return pre[last:] if last != -1 else pre

def _san_candidates(
    board: chess.Board, text: str
) -> List[Tuple[int, int, str, str]]:
    """Every legally-parseable SAN token in ``text`` as (start, end, san, uci)."""
    out: List[Tuple[int, int, str, str]] = []
    for m in _SAN_RE.finditer(text):
        try:
            mv = board.parse_san(m.group(1))
        except ValueError:
            continue
        out.append((m.start(), m.end(), board.san(mv), mv.uci()))
    return out

def _is_avoid_framed(text: str, start: int, end: int) -> bool:
    """True if the SAN at ``[start, end)`` is framed as a move to AVOID / not the
    coach's pick: an avoid cue ("rather than", "like", "instead of"), a
    hypothetical/continuation ("if White plays", "after ... "), a bare square
    reference ("the rook goes to h3", "the g4-pawn"), or a dismissed concession."""
    pre = _clause_before(text, start)
    if _AVOID_CUE_RE.search(pre) or _HYPO_CUE_RE.search(pre):
        return True
    tok = text[start:end]
    if tok[:1] in "abcdefgh" and "x" not in tok:  # bare pawn move -> maybe a square ref
        if _COORD_BEFORE_RE.search(text[max(0, start - 6) : start]):
            return True
        if _COORD_AFTER_RE.search(text[end : end + 8]):
            return True
    if _CONCESSION_AFTER_RE.search(text[end : end + 26]):
        return True
    return False

def _endorsed_indices(
    text: str, cands: Sequence[Tuple[int, int, str, str]]
) -> set:
    """Indices of candidates the coach explicitly ENDORSES as its own pick — named
    right after an endorsement cue ("I'd play X", "the move: X"), or in the
    imperative "Play X again" pattern. The endorsement is LOCALIZED: the move must
    sit right after the cue with no avoid/hypothetical framing in between, so
    "...such as Nc3, the move is Nf3" endorses Nf3 (the cue is right before it)
    without being fooled by the earlier "such as". Because it is this tightly
    scoped, an endorsed move is trusted even when the wider sentence opened with a
    list/avoid cue — that is how a genuine pick after an avoided list is recovered.
    May be the student's own move (a coach can endorse a move it agrees with)."""
    endorsed: set = set()
    for cue in _ENDORSE_CUE_RE.finditer(text):
        lo, hi = cue.end(), cue.end() + 16
        for i, (s, _e, _san, _uci) in enumerate(cands):
            if lo <= s <= hi:
                between = text[cue.end() : s]
                if not (_AVOID_CUE_RE.search(between) or _HYPO_CUE_RE.search(between)):
                    endorsed.add(i)
                break
    for i, (s, e, _san, _uci) in enumerate(cands):
        if "again" in text[e : e + 10].lower() and "play" in text[max(0, s - 8) : s].lower():
            endorsed.add(i)
    return endorsed

def pick_recommendation(
    text: str,
    board: chess.Board,
    student_uci: str,
    accept: Callable[[str], bool],
) -> Optional[Tuple[str, str]]:
    """Return (SAN, UCI) of the coach's ACTUAL recommended move, or ``None``.

    ``accept(uci) -> bool`` is the caller's move filter (in the Stockfish sound
    pool for the shipped coach; any legal move for the honest metrics extractor).
    In order of preference the recommendation is:

    1. a move named right after an EXPLICIT endorsement cue ("I'd play X",
       "the move: X", "Play X again") — even when it equals the student's own
       move, because the coach can endorse a good move it agrees with;
    2. otherwise the first non-student move that is NOT framed as one to avoid;
    3. otherwise the student's own move if it is stated approvingly (non-avoid).

    Moves framed as things to AVOID ("rather than ...b2+", "instead of Rh8",
    "a forcing move like ...b2+"), opponent replies / continuations ("if White
    plays Ke4", "after d2+ ...f3"), and bare square references ("the rook goes to
    h3") are never chosen. That avoid-framing class is the bug this fixes: the old
    scanner grabbed the first non-student legal SAN, which — when the coach agreed
    with the student's move — was often a move the coach told the student NOT to
    play, corrupting the shown move and faking a tier "fork".
    """
    cands = [t for t in _san_candidates(board, text) if accept(t[3])]
    if not cands:
        return None
    avoid = [_is_avoid_framed(text, s, e) for (s, e, _san, _uci) in cands]
    endorsed = _endorsed_indices(text, cands)

    for i, (_s, _e, san, uci) in enumerate(cands):  # 1) explicitly endorsed pick
        if i in endorsed:  # endorsement is already localized (avoid-free span)
            return san, uci
    for i, (_s, _e, san, uci) in enumerate(cands):  # 2) first clean alternative
        if uci != student_uci and not avoid[i]:
            return san, uci
    for i, (_s, _e, san, uci) in enumerate(cands):  # 3) approvingly-stated student move
        if uci == student_uci and not avoid[i]:
            return san, uci
    return None


# --- from src/eval/evaluate.py (verbatim) ---
def _extract_json_object(text: str) -> Optional[Dict[str, Any]]:
    """Return the first balanced ``{...}`` JSON object in ``text``, or ``None``."""
    start = text.find("{")
    while start != -1:
        depth = 0
        for i in range(start, len(text)):
            ch = text[i]
            if ch == "{":
                depth += 1
            elif ch == "}":
                depth -= 1
                if depth == 0:
                    try:
                        obj = json.loads(text[start : i + 1])
                    except json.JSONDecodeError:
                        break
                    if isinstance(obj, dict):
                        return obj
                    break
        start = text.find("{", start + 1)
    return None

def _legal_uci(board: chess.Board, san_token: str) -> Optional[str]:
    """Return the UCI of ``san_token`` if it is a legal SAN in ``board``, else None."""
    try:
        move = board.parse_san(san_token)
    except (ValueError, AssertionError):
        return None
    return move.uci()

def extract_recommended_move(
    text: str, fen: str, student_uci: str
) -> Tuple[Optional[str], Optional[str]]:
    """Best-effort extraction of the coach's recommended move as ``(san, uci)``.

    Strategy (deterministic; shares its avoid-framing-aware core with the shipped
    coach in :func:`src.teacher.coach_gate.pick_recommendation`):
      1. If the output embeds a JSON object with ``recommended_move_uci`` /
         ``recommended_move_san`` (a tuned model may emit structured output), use
         it when legal.
      2. Otherwise return the coach's ACTUAL recommended move via
         :func:`pick_recommendation` — the move after an explicit "I'd play X" /
         "the move: X" / "Play X again" cue (even if it equals the student's own
         move, e.g. the student already played the best move), else the first
         non-student move that is NOT framed as one to avoid.

    Crucially, a move the coach frames as one to AVOID ("rather than ...b2+",
    "a forcing move like ...b2+"), an opponent reply / continuation ("if White
    plays Ke4", "after d2+ ...f3"), or a bare square reference ("the rook goes to
    h3") is never mistaken for the recommendation. Any legal move is accepted (the
    move need not be sound — soundness is scored separately), so an unsound pick
    is reported honestly rather than hidden.

    Returns ``(None, None)`` when no recommended move can be recovered.
    """
    board = chess.Board(fen)

    obj = _extract_json_object(text)
    if obj:
        uci = obj.get("recommended_move_uci")
        san = obj.get("recommended_move_san")
        if isinstance(uci, str):
            try:
                mv = chess.Move.from_uci(uci.strip())
                if mv in board.legal_moves:
                    return board.san(mv), mv.uci()
            except ValueError:
                pass
        if isinstance(san, str):
            got = _legal_uci(board, san.strip())
            if got is not None:
                return board.san(chess.Move.from_uci(got)), got

    picked = pick_recommendation(text, board, student_uci, accept=lambda _u: True)
    return picked if picked is not None else (None, None)

print('vendored extractor ready (byte-identical to the repo)')


In [ ]:
import re
from statistics import mean
TIERS = ('beginner', 'intermediate', 'advanced')

def _strip_think(text):
    # Identical to scripts/stage4_eval_v6dpo2.py (matches how the published gens were cleaned).
    text = re.sub(r'<think>.*?</think>', '', text or '', flags=re.DOTALL)
    return text.replace('<think>', '').replace('</think>', '').strip()

def _extract(output, fen, student_uci):
    _san, uci = extract_recommended_move(output or '', fen, student_uci or '')
    return uci

def score_condition(gens_by_id, rows):
    # rows: list of prompt records (targets). gens_by_id: {id: output_text}.
    # Mirrors scripts/stage4_eval_v6dpo2.score_condition exactly.
    by_tier = {t: [0, 0] for t in TIERS}
    sound = [0, 0]; named = [0, 0]; fmt = [0, 0]
    preds_by_pos = {}; canon_by_pos = {}
    n = 0
    for r in rows:
        out = gens_by_id.get(r['id'])
        if out is None:
            continue
        n += 1
        tier = r['tier']
        uci = _extract(out, r['fen'], r.get('student_uci') or '')
        if tier in by_tier:
            by_tier[tier][1] += 1
            if uci and uci == r.get('canonical_uci'):
                by_tier[tier][0] += 1
        sound[1] += 1
        if uci and uci in set(r.get('sound_ucis', [])):
            sound[0] += 1
        named[1] += 1
        if uci:
            named[0] += 1
        fmt[1] += 1
        if uci and ("I'd play" in out or "I\u2019d play" in out) and 'Takeaway:' in out:
            fmt[0] += 1
        preds_by_pos.setdefault(r['pos_id'], {})[tier] = uci
        canon_by_pos.setdefault(r['pos_id'], {})['beginner'] = r.get('canonical_beginner_uci')
        canon_by_pos[r['pos_id']]['advanced'] = r.get('canonical_advanced_uci')
    per_tier = {t: (by_tier[t][0] / by_tier[t][1]) for t in TIERS if by_tier[t][1]}
    diff = dist = 0
    for pid, cd in canon_by_pos.items():
        cb, ca = cd.get('beginner'), cd.get('advanced')
        if not (cb and ca and cb != ca):
            continue
        diff += 1
        mb = preds_by_pos.get(pid, {}).get('beginner')
        ma = preds_by_pos.get(pid, {}).get('advanced')
        if mb and ma and mb != ma:
            dist += 1
    return {
        'tier_policy_match': round(mean(per_tier.values()), 4) if per_tier else 0.0,
        'per_tier': {t: round(v, 4) for t, v in per_tier.items()},
        'per_tier_counts': {t: by_tier[t] for t in TIERS if by_tier[t][1]},
        'move_sound': round(sound[0] / sound[1], 4) if sound[1] else 0.0,
        'named_rate': round(named[0] / named[1], 4) if named[1] else 0.0,
        'format_rate': round(fmt[0] / fmt[1], 4) if fmt[1] else 0.0,
        'distinct_rate': round(dist / diff, 4) if diff else 0.0,
        'distinct_counts': [dist, diff],
        'n': n,
    }
print('scorer ready')

## Step 6 - Load Qwen3-32B (4-bit) + the v6-dpo2 LoRA (Unsloth)

This downloads the ~39 GB 4-bit base on first run (cached afterwards) and attaches the LoRA. On an A100-80GB this takes a few minutes.

*Why Unsloth and not vLLM: the published numbers come from this exact Unsloth + Hugging Face `generate()` path, including `no_repeat_ngram_size=4`, which vLLM's sampler does not replicate. Using the same path is what makes the numbers comparable.*

In [ ]:
import torch
from unsloth import FastLanguageModel
from peft import PeftModel

model, tok = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL, max_seq_length=MAX_SEQ_LEN, load_in_4bit=True, dtype=None,
)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
model = PeftModel.from_pretrained(model, ADAPTER_DIR, adapter_name='v6_dpo2')
FastLanguageModel.for_inference(model)
model.set_adapter('v6_dpo2')
print('loaded base + v6-dpo2 adapter | device:', next(model.parameters()).device)

## Step 7 - Generate all 803 x 3 (batched, greedy, checkpointed)

Appends each finished batch to Drive and skips anything already done. **If the runtime disconnects or you run out of units, just re-run this cell** (after re-running Steps 1-6 to reload the model); it resumes where it left off. Greedy decoding is deterministic, so a re-run of any lost in-flight batch is identical.

In [ ]:
import json, os, time, torch

def render(r):
    return tok.apply_chat_template(
        [{'role': 'system', 'content': r['grounded_system']},
         {'role': 'user',   'content': r['grounded_user']}],
        tokenize=False, add_generation_prompt=True, enable_thinking=False,
    )

done = set()
if RESUME and os.path.exists(GENS_PATH):
    for line in open(GENS_PATH, encoding='utf-8'):
        line = line.strip()
        if line:
            try:
                done.add(json.loads(line)['id'])
            except Exception:
                pass
todo = [r for r in ROWS if r['id'] not in done]
print('already done: %d | to generate: %d | total: %d' % (len(done), len(todo), len(ROWS)))

t0 = time.time()
fh = open(GENS_PATH, 'a', encoding='utf-8')
try:
    for i in range(0, len(todo), EVAL_BATCH):
        batch = todo[i:i + EVAL_BATCH]
        texts = [render(r) for r in batch]
        tok.padding_side = 'left'; tok.truncation_side = 'left'
        enc = tok(texts, return_tensors='pt', padding=True, truncation=True,
                  max_length=MAX_SEQ_LEN).to('cuda')
        try:
            with torch.no_grad():
                gen = model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                                     repetition_penalty=REPETITION_PENALTY,
                                     no_repeat_ngram_size=NO_REPEAT_NGRAM,
                                     pad_token_id=tok.pad_token_id)
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            print('OOM at batch offset %d. Lower EVAL_BATCH in the config cell, re-run Step 6 '
                  'if needed, then re-run this cell to resume.' % i)
            raise
        for r, g, inp in zip(batch, gen, enc['input_ids']):
            raw = _strip_think(tok.decode(g[inp.shape[0]:], skip_special_tokens=True))
            fh.write(json.dumps({'id': r['id'], 'output': raw}, ensure_ascii=False) + '\n')
        fh.flush(); os.fsync(fh.fileno())
        n_done = len(done) + i + len(batch)
        el = time.time() - t0
        rate = (i + len(batch)) / max(1e-9, el)
        eta_min = (len(todo) - (i + len(batch))) / max(1e-9, rate) / 60.0
        print('  done %d/%d | %.2f gen/s | ETA %.0f min' % (n_done, len(ROWS), rate, eta_min), flush=True)
finally:
    fh.close()
print('generation cell finished. If it stopped early, just re-run it to resume.')

## Step 8 - Score + final table (v6-dpo2 803 vs the field, with the self-validation)

In [ ]:
import json, os

gens_by_id = {}
if os.path.exists(GENS_PATH):
    for line in open(GENS_PATH, encoding='utf-8'):
        line = line.strip()
        if line:
            o = json.loads(line)
            gens_by_id[o['id']] = o['output']

covered = sum(1 for r in ROWS if r['id'] in gens_by_id)
print('coverage: %d/%d scenarios generated%s\n' % (
    covered, len(ROWS), '' if covered == len(ROWS) else '  (PARTIAL - resume Step 7 for the full 803)'))

full = score_condition(gens_by_id, ROWS)
val  = score_condition(gens_by_id, [r for r in ROWS if r['is_val']])

def show(title, s):
    pt = s['per_tier']
    print(title)
    print('  tier-policy match : %.4f   (B %.4f / I %.4f / A %.4f)' % (
        s['tier_policy_match'], pt.get('beginner', 0), pt.get('intermediate', 0), pt.get('advanced', 0)))
    print('  move-sound        : %.4f' % s['move_sound'])
    print('  distinct-per-level: %.4f  (%d/%d)' % (s['distinct_rate'], s['distinct_counts'][0], s['distinct_counts'][1]))
    print('  names-a-move      : %.4f' % s['named_rate'])
    print('  format            : %.4f' % s['format_rate'])
    print('  n                 : %d' % s['n'])

print('=' * 72)
print('v6-dpo2 - FULL 803 x 3 (fresh corrected-v6 grounding, greedy Stage-4 decode)')
print('=' * 72)
show('', full)

print('\n' + '=' * 72)
print('SELF-VALIDATION - 120 held-out TEST subset (must reproduce published Stage-4)')
print('=' * 72)
show('', val)
ok = abs(val['tier_policy_match'] - 0.8917) < 2e-3 if val['n'] == 360 else None
if val['n'] < 360:
    print('  (only %d/360 TEST scenarios done so far - finish Step 7 to validate)' % val['n'])
else:
    print('  published Stage-4 v6-dpo2 (120 TEST): tier 0.8917 | B 0.8583 / I 0.8417 / A 0.9750 | sound 0.9833 | distinct 75/76')
    print('  SELF-VALIDATION: %s' % ('PASS - method reproduces the published numbers' if ok
                                     else 'CHECK - subset differs from 0.8917; investigate before reporting'))

print('\n' + '=' * 72)
print('CONTEXT (same method: fresh corrected-v6 grounding, 120 held-out TEST)')
print('  from RESULTS_STAGE4_CORRECTED.md - do NOT cross-compare to the 803 field table,')
print('  which is a re-score of OLD (v4-era-grounding) generations, a different scope.')
print('=' * 72)
print('  OURS-v4 (shipped) : tier 0.8611   (B 0.858 / I 0.750 / A 0.975)')
print('  OURS-v6-dpo       : tier 0.8806   (B 0.858 / I 0.808 / A 0.975)')
print('  OURS-v6-dpo2      : tier 0.8917   (B 0.858 / I 0.842 / A 0.975)   <- this notebook, 120-TEST subset')
print('  BASE (untuned)    : tier 0.4278   (B 0.442 / I 0.408 / A 0.433)')

report = {
    'model': 'chess-coach-32b-v6-dpo2',
    'base_model': BASE_MODEL,
    'benchmark': 'scenarios_v6 (corrected labels), FULL 803 positions x 3 tiers',
    'grounding': 'byte-identical to Stage-4 (facts + render_user_prompt + format instruction)',
    'decode': {'do_sample': False, 'repetition_penalty': REPETITION_PENALTY,
               'no_repeat_ngram_size': NO_REPEAT_NGRAM, 'max_new_tokens': MAX_NEW_TOKENS},
    'coverage': covered, 'n_total': len(ROWS),
    'scores_full_803': full,
    'scores_holdout_120_test': val,
    'self_validation_pass': ok,
}
with open(SCORES_PATH, 'w', encoding='utf-8') as fh:
    json.dump(report, fh, indent=2)
print('\nwrote metrics -> %s' % SCORES_PATH)

## Step 9 (optional) - Publish the results to Hugging Face

Saves the generations + metrics to a results dataset so the 803 v6-dpo2 row is archived. Skip if you only need the printed table (already saved to Drive).

In [ ]:
# Optional: push generations + scores to a HF dataset (needs a WRITE token).
DO_UPLOAD = False   # set True to run
RESULTS_DATASET = 'khoilamalphaai/chess-coach-803-eval-prompts'  # or a dedicated results repo

if DO_UPLOAD:
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    api.upload_file(path_or_fileobj=GENS_PATH,
                    path_in_repo='results/v6dpo2_gens_803.jsonl',
                    repo_id=RESULTS_DATASET, repo_type='dataset',
                    commit_message='Add v6-dpo2 full-803 generations (Colab)')
    api.upload_file(path_or_fileobj=SCORES_PATH,
                    path_in_repo='results/v6dpo2_scores_803.json',
                    repo_id=RESULTS_DATASET, repo_type='dataset',
                    commit_message='Add v6-dpo2 full-803 scores (Colab)')
    print('uploaded results to', RESULTS_DATASET)
else:
    print('DO_UPLOAD is False - results are saved to Drive at', SCORES_PATH)

## Notes on honesty and comparability

- **Same grounding, same extractor, same decode.** The prompts are byte-identical to the
  Stage-4 eval (verified offline: the 360 held-out TEST prompts match `stage4_eval_inputs.jsonl`
  exactly), the extractor is copied verbatim from `src.eval.evaluate`, and the greedy decode
  matches `scripts/stage4_eval_v6dpo2.py`. That is what makes the 803 row legitimately
  comparable to `RESULTS_STAGE4_CORRECTED.md`.
- **No verify-gate is applied during scoring, on purpose.** The published Stage-4 and 803
  field tier-policy numbers are computed on single greedy generations (no live
  verify-and-regenerate loop). The gate is a serving-time faithfulness floor applied equally
  to every model (`RESULTS_FULL_EVAL_803.md` section 4), not part of tier-policy scoring, so
  reproducing the eval means NOT gating. Adding the gate would move the numbers away from the
  comparable tables.
- **Scope.** These are corrected-v6 fresh-grounding numbers. The 803 field table in
  `RESULTS_FULL_EVAL_803.md` is a re-score of OLD generations against the corrected labels,
  a different scope; compare v6-dpo2 here to the fresh-grounding Stage-4 numbers, and treat the
  field table as ranking context only.
- **Tier-policy match** is agreement with a preregistered project rule (learnability), not a
  claim of validated pedagogy.